# 09 — Final held-out evaluation

**Objective.** Measure the selected system once, on documents never used in development. B1 is ColQwen2 top 4 → `gpt-6-luna` with contract b1. The evaluation:
- estimates B1's accuracy (**C1**);
- re-tests the project's central result, visual retrieval against OCR + BM25 retrieval with the same reader (**C2**, confirmatory);
- reports E9's adaptive policy A1 as a secondary contrast (**C3**).

The protocol is frozen in `experiments/specs/final_evaluation_v1.json` (commit `1c54a3e`). The owner approved it on 2026-11-25, before any test-pool document was downloaded.

**Stages and modes** (`VISUALDOC_FINAL_MODE`):

| Mode | What it does | Reads questions? | Paid? |
|---|---|---|---|
| `offline` (default) | reports the state of every stage; evaluates a completed run | only once the index is frozen | no |
| `index` | downloads the 111 pool PDFs, extracts text (OCR below 30 words), renders 768-px pages, runs the E5 parity gate, then encodes every page with ColQwen2 (resumable) | **no** | no |
| `queries` | after the index manifest is committed: loads the pool questions and encodes them | yes | no |
| `smoke` / `full` | the reader runs (smoke on development questions only) | yes | yes |

**Inputs.**
- The frozen spec.
- `resources/mmlongbench_doc_documents.json` (document names).
- The MMLongBench-Doc PDFs at revision `2ff6aa92…`, downloaded here.
- The ColQwen2 snapshot `ddc07d23…`, in the Hugging Face cache outside the repository.
- The E5 encoder configuration and cache, used for the parity gate.

**Outputs.**
- `data/processed/mmlongbench_doc/final/` (document and page tables, the encoding log, the index manifest).
- `data/cache/final/` (page embeddings).
- Later, the run and its evaluation.

**Next step.** Commit the index manifest; then the `queries` mode.

## Method (indexing stage)

**Same pipeline as E5, unchanged.** Every page is represented in two ways:
- **Text:** the native text layer, or full-page Tesseract OCR at 150 dpi when it has fewer than 30 words. This text feeds BM25 for T1.
- **Image:** the whole page fitted in a 768 × 1536 box as an RGB PNG. This is the image ColQwen2 indexes and the reader sees.

ColQwen2 maps an image to a matrix of 128-dimensional vectors, one per visual token, and a question to one vector per query token. Retrieval scores a page by late interaction (MaxSim): $s(q,d)=\sum_{i}\max_j \langle q_i, d_j\rangle$.

**Two passes, now in chunks.** On an 8 GB machine, E5 ran the vision tower over every page first (pass 1), then the language model with the vision tower replaced by the saved features (pass 2). E5 proved this bitwise equal to the single pass.
- Pass-1 features take about 1.8 MB per page.
- For about 5,300 pages that would mean about 9.5 GB of transient disk.
- So the passes alternate over chunks of 200 pages. Each embedding is computed by exactly the same operations, so this changes only when the model is reloaded, not what is computed.

**Parity gate (frozen spec).** Before any pool page is encoded, two development pages and one development question are re-encoded from scratch. All three must be **bitwise equal** to the frozen E5 cache. This proves that the weights, processor, library versions and the chunked two-pass procedure reproduce E5 exactly. Otherwise the notebook stops.

**No question is read in this stage.** Indexing depends only on the documents. The pool questions are loaded only after the index manifest (every page's image hash and embedding hash) is committed, so retrieval cannot be adapted to the questions.

In [1]:
# Setup: paths, the frozen spec (must equal its freeze commit), mode, and small file utilities.
from pathlib import Path
from types import SimpleNamespace
import gc
import hashlib
import io
import json
import os
import subprocess
import time

import numpy as np
import pandas as pd
import pymupdf
from IPython.display import display

pd.set_option("display.width", 170)


def find_project_root(start=Path.cwd()):
    for candidate in (start, *start.parents):
        if (candidate / "pyproject.toml").is_file():
            return candidate
    raise FileNotFoundError("Could not find the repository root")


PROJECT_ROOT = find_project_root()
SPEC_PATH = PROJECT_ROOT / "experiments" / "specs" / "final_evaluation_v1.json"
SPEC_FREEZE_COMMIT = "1c54a3e"
DOC_LIST_PATH = PROJECT_ROOT / "resources" / "mmlongbench_doc_documents.json"
MMLONG_DIR = PROJECT_ROOT / "data" / "external" / "mmlongbench_doc"
PDF_DIR = MMLONG_DIR / "documents"
TESSDATA_DIR = PROJECT_ROOT / "resources" / "tessdata"
IMAGE_DIR = PROJECT_ROOT / "data" / "cache" / "mmlongbench_doc" / "page_images" / "short768-long1536"
FINAL_DIR = PROJECT_ROOT / "data" / "processed" / "mmlongbench_doc" / "final"
E5_CACHE_DIR = PROJECT_ROOT / "data" / "cache" / "e5" / "colqwen2-v1.0-hf@ddc07d2317c8" / "af02fac2cb9b"
E5_FROZEN = PROJECT_ROOT / "experiments" / "frozen" / "e5"

REQUIRED = {SPEC_PATH: "restore the frozen spec from git", DOC_LIST_PATH: "restore resources/ from git",
            TESSDATA_DIR / "eng.traineddata": "the OCR model is tracked in git: restore resources/tessdata",
            E5_FROZEN / "SHA256SUMS": "restore experiments/frozen/e5 from git"}
missing = [f"{p.relative_to(PROJECT_ROOT)} ({fix})" for p, fix in REQUIRED.items() if not p.is_file()]
assert not missing, "Notebook 09 inputs missing:\n  " + "\n  ".join(missing)

git = lambda *args: subprocess.run(["git", *args], cwd=PROJECT_ROOT, capture_output=True, check=True).stdout
assert git("show", f"{SPEC_FREEZE_COMMIT}:experiments/specs/final_evaluation_v1.json") == SPEC_PATH.read_bytes(), \
    "the final-evaluation spec changed after its freeze commit: that is a new condition"
SPEC = json.loads(SPEC_PATH.read_text())
REVISION = SPEC["population"]["revision"]
MODE = os.environ.get("VISUALDOC_FINAL_MODE", "offline")
assert MODE in {"offline", "index", "queries", "smoke", "full"}, f"unknown VISUALDOC_FINAL_MODE {MODE!r}"
SETTINGS = {"ocr_min_native_words": 30, "ocr_dpi": 150, "image_short_px": 768, "image_long_px": 1536,
            "bm25_k1": 1.5, "bm25_b": 0.75, "encode_chunk_pages": 200}
FINAL_DIR.mkdir(parents=True, exist_ok=True)


def sha256_bytes(data):
    return hashlib.sha256(data).hexdigest()


def sha256_file(path, chunk=1 << 24):
    digest = hashlib.sha256()
    with open(path, "rb") as handle:
        for block in iter(lambda: handle.read(chunk), b""):
            digest.update(block)
    return digest.hexdigest()


def atomic_write_bytes(path, data):
    """Write to a temporary sibling, fsync, then rename: an interruption never leaves a half-written file."""
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_name(f".{path.name}.tmp-{os.getpid()}")
    with open(tmp, "wb") as handle:
        handle.write(data)
        handle.flush()
        os.fsync(handle.fileno())
    os.replace(tmp, path)


def remove_stale_temporaries(folder):
    stale = list(Path(folder).glob(".*.tmp-*")) if Path(folder).is_dir() else []
    for path in stale:
        path.unlink()
    return len(stale)


print(f"Spec equals commit {SPEC_FREEZE_COMMIT}. Mode: {MODE}. Outputs: {FINAL_DIR.relative_to(PROJECT_ROOT)}")

Spec equals commit 1c54a3e. Mode: offline. Outputs: data/processed/mmlongbench_doc/final


In [2]:
# The population, from document names only (frame metadata): 135 - 20 development - 4 touched = 111. No question is read.
doc_list = json.loads(DOC_LIST_PATH.read_text())
assert doc_list["revision"] == REVISION
ALL_DOCS = sorted(doc_list["documents"])
assert len(ALL_DOCS) == 135

e5_spec = json.loads((PROJECT_ROOT / "experiments" / "specs" / "e5_visual_retrieval_v1.json").read_text())
HELDOUT14 = sorted(e5_spec["populations"]["held_out"]["selected_documents"])
DEV6 = sorted(pd.read_csv(E5_FROZEN / "tables" / "per_question.csv").query("population == 'development'")["doc_id"].unique())
TOUCHED = sorted(SPEC["population"]["excluded_touched_documents"])
assert len(HELDOUT14) == 14 and len(DEV6) == 6 and len(TOUCHED) == 4
assert not (set(HELDOUT14) | set(DEV6)) & set(TOUCHED) and set(HELDOUT14) | set(DEV6) | set(TOUCHED) <= set(ALL_DOCS)
POOL_DOCS = sorted(set(ALL_DOCS) - set(HELDOUT14) - set(DEV6) - set(TOUCHED))
assert len(POOL_DOCS) == SPEC["population"]["documents"] == 111
print(f"Population: {len(POOL_DOCS)} documents (135 - 20 development - 4 touched). No question has been read.")

Population: 111 documents (135 - 20 development - 4 touched). No question has been read.


In [3]:
# Download the 111 pool PDFs at the pinned revision (index mode), and record size, SHA-256 and page count.
DOCS_TABLE = FINAL_DIR / "pool_documents.csv"


def local_pdf(doc_id):
    return PDF_DIR / doc_id


def materialised(path):
    """A real local file (not an iCloud placeholder that would read as empty)."""
    return path.is_file() and path.stat().st_size > 0 and not (os.stat(path).st_flags & 0x40000000)


if MODE == "index":
    from huggingface_hub import hf_hub_download
    rows, started = [], time.perf_counter()
    for i, doc_id in enumerate(POOL_DOCS, 1):
        if not materialised(local_pdf(doc_id)):
            # A file left by an earlier download may be an evicted iCloud placeholder (it reads as empty): force a fresh copy.
            hf_hub_download("yubo2333/MMLongBench-Doc", f"documents/{doc_id}", repo_type="dataset", revision=REVISION, local_dir=MMLONG_DIR,
                            force_download=local_pdf(doc_id).exists())
        path = local_pdf(doc_id)
        assert materialised(path), f"{doc_id}: download did not produce a local file"
        try:
            with pymupdf.open(path) as pdf:
                pages, encrypted = pdf.page_count, pdf.needs_pass
        except Exception as error:  # recorded, never silently dropped
            pages, encrypted = 0, None
        rows.append({"doc_id": doc_id, "bytes": path.stat().st_size, "sha256": sha256_file(path), "pages": pages, "encrypted": encrypted})
        if i % 20 == 0 or i == len(POOL_DOCS):
            print(f"  {i}/{len(POOL_DOCS)} documents, {time.perf_counter() - started:.0f} s", flush=True)
    atomic_write_bytes(DOCS_TABLE, pd.DataFrame(rows).to_csv(index=False).encode())
if DOCS_TABLE.is_file():
    pool_documents = pd.read_csv(DOCS_TABLE)
    assert sorted(pool_documents["doc_id"]) == POOL_DOCS
    unreadable = pool_documents[(pool_documents["pages"] < 1) | (pool_documents["encrypted"] == True)]
    print(f"{len(pool_documents)} pool PDFs: {pool_documents['bytes'].sum() / 1e9:.2f} GB, {int(pool_documents['pages'].sum())} pages "
          f"(per document: median {pool_documents['pages'].median():.0f}, max {pool_documents['pages'].max()}); unreadable: {len(unreadable)}.")
    display(pd.cut(pool_documents["pages"], [0, 50, 200, 10_000], labels=["<= 50", "51-200", "> 200"]).value_counts().sort_index().to_frame("documents"))
else:
    pool_documents = None
    print("Pool PDFs not downloaded yet (run with VISUALDOC_FINAL_MODE=index).")

Pool PDFs not downloaded yet (run with VISUALDOC_FINAL_MODE=index).


In [4]:
# Text (notebook 00's rule) and the 768-px render (E1 profile) of every pool page; one parquet per document, resumable.
PAGES_DIR = FINAL_DIR / "pages"


def page_text_record(page):
    native_text = page.get_text("text", sort=True)
    native_words = len(native_text.split())
    ocr_used = native_words < SETTINGS["ocr_min_native_words"]
    if ocr_used:
        textpage = page.get_textpage_ocr(language="eng", dpi=SETTINGS["ocr_dpi"], full=True, tessdata=str(TESSDATA_DIR))
        text = page.get_text("text", textpage=textpage, sort=True)
    else:
        text = native_text
    return {"native_words": native_words, "ocr_used": ocr_used, "text": text}


def render_answering_png(page):
    zoom = min(SETTINGS["image_short_px"] / min(page.rect.width, page.rect.height),
               SETTINGS["image_long_px"] / max(page.rect.width, page.rect.height))
    return page.get_pixmap(matrix=pymupdf.Matrix(zoom, zoom), alpha=False).tobytes("png")


def image_path(doc_id, page_number):
    return IMAGE_DIR / Path(doc_id).stem / f"page_{page_number:03d}.png"


# Determinism check on development pages first: a fresh render must equal E5's frozen bytes.
e5_manifest = pd.read_csv(E5_FROZEN / "tables" / "render_manifest_heldout.csv")
render_checked = []
for doc_id, group in e5_manifest.groupby("doc_id"):
    if materialised(local_pdf(doc_id)) and len(render_checked) < 2:  # iCloud placeholders cannot be read offline
        with pymupdf.open(local_pdf(doc_id)) as pdf:
            for row in group.head(3).itertuples():
                assert sha256_bytes(render_answering_png(pdf[row.page - 1])) == row.sha256, f"render differs from E5 for {doc_id} p.{row.page}"
        render_checked.append(doc_id)
assert render_checked, "no development PDF is available locally to check rendering determinism"
print(f"Rendering reproduces E5's frozen page bytes on 3 pages of each of {len(render_checked)} development documents.")

if MODE == "index":
    remove_stale_temporaries(PAGES_DIR)
    started, done = time.perf_counter(), 0
    for doc_id in POOL_DOCS:
        target = PAGES_DIR / f"{Path(doc_id).stem}.parquet"
        if target.is_file():
            continue
        rows = []
        with pymupdf.open(local_pdf(doc_id)) as pdf:
            for index, page in enumerate(pdf):
                png, out = render_answering_png(page), image_path(doc_id, index + 1)
                if not out.is_file() or out.read_bytes() != png:
                    atomic_write_bytes(out, png)
                width, height = int.from_bytes(png[16:20], "big"), int.from_bytes(png[20:24], "big")
                rows.append({"doc_id": doc_id, "page": index + 1, **page_text_record(page), "width_px": width, "height_px": height,
                             "sha256": sha256_bytes(png), "image_path": str(out.relative_to(PROJECT_ROOT))})
        buffer = io.BytesIO()
        pd.DataFrame(rows).to_parquet(buffer, index=False)
        atomic_write_bytes(target, buffer.getvalue())
        done += 1
        print(f"  pages: {doc_id[:50]} ({len(rows)} pages), {done} new documents, {time.perf_counter() - started:.0f} s", flush=True)

page_files = sorted(PAGES_DIR.glob("*.parquet")) if PAGES_DIR.is_dir() else []
pool_pages = pd.concat([pd.read_parquet(p) for p in page_files], ignore_index=True) if page_files else None
if pool_pages is not None:
    print(f"Page records for {pool_pages['doc_id'].nunique()}/{len(POOL_DOCS)} documents: {len(pool_pages)} pages, "
          f"OCR on {int(pool_pages['ocr_used'].sum())}, {pool_pages['sha256'].nunique()} distinct images.")

Rendering reproduces E5's frozen page bytes on 3 pages of each of 2 development documents.


In [5]:
# The ColQwen2 encoder: configuration identical to E5's (checked field by field), weights verified by SHA-256.
ENCODE_MODES = {"index", "queries"}
E5_CONFIG = json.loads((E5_CACHE_DIR / "config.json").read_text()) if (E5_CACHE_DIR / "config.json").is_file() else None
FINAL_CACHE = PROJECT_ROOT / "data" / "cache" / "final" / "colqwen2-v1.0-hf@ddc07d2317c8" / "af02fac2cb9b"
ENCODING_LOG = FINAL_DIR / "encoding_log.csv"

if MODE in ENCODE_MODES:
    import torch
    import transformers
    from huggingface_hub import snapshot_download
    from PIL import Image
    from safetensors.torch import load_file, save_file
    from transformers import ColQwen2ForRetrieval, ColQwen2Processor
    import psutil

    assert E5_CONFIG is not None, "the E5 encoder cache (data/cache/e5/...) is needed for the identity and parity checks"
    MODEL_DIR = Path(snapshot_download(E5_CONFIG["repo"], revision=E5_CONFIG["revision"], local_files_only=True))
    DEVICE, DTYPE, ATTENTION = "mps", torch.bfloat16, "sdpa"
    processor = ColQwen2Processor.from_pretrained(MODEL_DIR)
    ENCODER_CONFIG = {
        "repo": E5_CONFIG["repo"], "revision": E5_CONFIG["revision"], "weights_sha256": sha256_file(MODEL_DIR / "model.safetensors"),
        "transformers": transformers.__version__, "torch": torch.__version__, "device": DEVICE, "dtype": "bfloat16",
        "attn_implementation": ATTENTION, "batch_size": 1, "page_image": "E1 profile: 768 x 1536 box, RGB PNG",
        "preprocessor_config_sha256": sha256_file(MODEL_DIR / "preprocessor_config.json"),
        "tokenizer_sha256": sha256_file(MODEL_DIR / "tokenizer.json"),
        "visual_prompt": processor.visual_prompt_prefix, "query_prefix": processor.query_prefix,
        "query_augmentation": f"10 x {processor.query_augmentation_token}",
        "embedding_rule": "model(**batch).embeddings; rows with attention_mask 0 removed; stored bf16",
    }
    differences = {k: (E5_CONFIG.get(k), ENCODER_CONFIG.get(k)) for k in set(E5_CONFIG) | set(ENCODER_CONFIG) if E5_CONFIG.get(k) != ENCODER_CONFIG.get(k)}
    assert not differences, f"the encoder is not E5's: {differences}"
    assert hashlib.sha256(json.dumps(ENCODER_CONFIG, sort_keys=True).encode()).hexdigest()[:12] == FINAL_CACHE.name
    for sub in ("pages", "queries", "vision_features", "parity"):
        (FINAL_CACHE / sub).mkdir(parents=True, exist_ok=True)
        remove_stale_temporaries(FINAL_CACHE / sub)
    atomic_write_bytes(FINAL_CACHE / "config.json", json.dumps(ENCODER_CONFIG, indent=1, sort_keys=True).encode())
    print(f"Encoder configuration equals E5's in all {len(ENCODER_CONFIG)} fields (weights {ENCODER_CONFIG['weights_sha256'][:12]}…). "
          f"Cache: {FINAL_CACHE.relative_to(PROJECT_ROOT)}")
else:
    print("Encoder not loaded (offline mode).")

Encoder not loaded (offline mode).


In [6]:
# The two passes of notebook 04, unchanged in substance, run over a list of page jobs (or questions) into a cache folder.
if MODE in ENCODE_MODES:
    process = psutil.Process()

    def atomic_save_tensors(tensors, path):
        tmp = path.with_name(f".{path.name}.tmp-{os.getpid()}")
        save_file(tensors, str(tmp))
        os.replace(tmp, path)

    def load_page_image(relative_path):
        with Image.open(PROJECT_ROOT / relative_path) as image:
            return image.convert("RGB")

    def unpadded_pixel_values(batch):
        # The library's own unpadding (ColQwen2ForRetrieval.forward): keep each image's h * w patch rows.
        grid = batch["image_grid_thw"]
        offsets = grid[:, 1] * grid[:, 2]
        mask = torch.arange(batch["pixel_values"].shape[1]).unsqueeze(0) < offsets.unsqueeze(1)
        return batch["pixel_values"][mask]

    class CachedVisionFeatures(torch.nn.Module):
        """Stands in for model.vlm.visual: returns the pass-1 output of the page being encoded."""
        def __init__(self):
            super().__init__()
            self.features = None

        def forward(self, pixel_values, grid_thw=None, return_dict=True, **kwargs):
            expected = int((grid_thw.prod(-1) // 4).sum())
            assert self.features is not None and self.features.shape[0] == expected, "cached features do not match this page"
            return SimpleNamespace(pooler_output=self.features)

    def encode_with(model, batch):
        embeddings = model(**batch).embeddings
        torch.mps.synchronize()
        keep = batch["attention_mask"][0].bool()
        return embeddings[0][keep].cpu().contiguous()

    def log_rows(rows):
        existing = ENCODING_LOG.read_bytes() if ENCODING_LOG.is_file() else b""
        atomic_write_bytes(ENCODING_LOG, existing + pd.DataFrame(rows).to_csv(index=False, header=not existing).encode())

    def pass1(jobs, cache):
        """Vision tower only, for pages without an embedding or saved features."""
        todo = [j for j in jobs if not (cache / "pages" / f"{j['sha256']}.safetensors").is_file()
                and not (cache / "vision_features" / f"{j['sha256']}.safetensors").is_file()]
        if not todo:
            return 0
        model = ColQwen2ForRetrieval.from_pretrained(MODEL_DIR, dtype=DTYPE, attn_implementation=ATTENTION).eval()
        visual = model.vlm.visual
        model.vlm.visual = None
        del model
        gc.collect()
        visual.to(DEVICE)
        rows = []
        with torch.inference_mode():
            for job in todo:
                t0 = time.perf_counter()
                batch = processor(images=[load_page_image(job["image_path"])], return_tensors="pt")
                features = visual(unpadded_pixel_values(batch).to(DEVICE), grid_thw=batch["image_grid_thw"].to(DEVICE), return_dict=True).pooler_output
                torch.mps.synchronize()
                atomic_save_tensors({"features": features.cpu().contiguous()}, cache / "vision_features" / f"{job['sha256']}.safetensors")
                rows.append({"pass": "vision", "key": job["sha256"], "doc_id": job["doc_id"], "page": job["page"],
                             "seconds": round(time.perf_counter() - t0, 3), "tokens": int(features.shape[0]), "rss_gb": round(process.memory_info().rss / 1e9, 2)})
        del visual
        gc.collect()
        torch.mps.empty_cache()
        log_rows(rows)
        return len(todo)

    def pass2(jobs, questions, cache):
        """The library's forward with the vision tower replaced by the pass-1 features; then any questions."""
        todo = [j for j in jobs if not (cache / "pages" / f"{j['sha256']}.safetensors").is_file()]
        qtodo = [q for q in questions if not (cache / "queries" / f"{sha256_bytes(q.encode())}.safetensors").is_file()]
        if not todo and not qtodo:
            return 0, 0
        model = ColQwen2ForRetrieval.from_pretrained(MODEL_DIR, dtype=DTYPE, attn_implementation=ATTENTION).eval()
        stub = CachedVisionFeatures()
        model.vlm.visual = stub
        gc.collect()
        model.to(DEVICE)
        rows = []
        with torch.inference_mode():
            for job in todo:
                t0 = time.perf_counter()
                batch = processor(images=[load_page_image(job["image_path"])], return_tensors="pt").to(DEVICE)
                feature_file = cache / "vision_features" / f"{job['sha256']}.safetensors"
                stub.features = load_file(str(feature_file))["features"].to(DEVICE)
                embedding = encode_with(model, batch)
                atomic_save_tensors({"embedding": embedding}, cache / "pages" / f"{job['sha256']}.safetensors")
                feature_file.unlink()  # embedding first, then its features
                rows.append({"pass": "language", "key": job["sha256"], "doc_id": job["doc_id"], "page": job["page"],
                             "seconds": round(time.perf_counter() - t0, 3), "tokens": int(embedding.shape[0]), "rss_gb": round(process.memory_info().rss / 1e9, 2)})
            stub.features = None
            for question in qtodo:
                t0 = time.perf_counter()
                embedding = encode_with(model, processor(text=[question], return_tensors="pt").to(DEVICE))
                atomic_save_tensors({"embedding": embedding}, cache / "queries" / f"{sha256_bytes(question.encode())}.safetensors")
                rows.append({"pass": "query", "key": sha256_bytes(question.encode()), "doc_id": "", "page": 0,
                             "seconds": round(time.perf_counter() - t0, 3), "tokens": int(embedding.shape[0]), "rss_gb": round(process.memory_info().rss / 1e9, 2)})
        del model, stub
        gc.collect()
        torch.mps.empty_cache()
        log_rows(rows)
        return len(todo), len(qtodo)

    print("Two-pass encoding functions defined (pass 1: vision tower; pass 2: language model with cached vision features).")

In [7]:
# Parity gate (frozen spec): two development pages and one development question, encoded from scratch, must equal E5 bit for bit.
PARITY_FILE = FINAL_DIR / "parity_gate.json"
if MODE in ENCODE_MODES:
    parity_pages = e5_manifest.groupby("doc_id").head(1).sort_values("doc_id").head(2)
    parity_jobs = [{"doc_id": r.doc_id, "page": int(r.page), "sha256": r.sha256, "image_path": r.image_path} for r in parity_pages.itertuples()]
    e8_first = json.loads((PROJECT_ROOT / "experiments" / "frozen" / "e8" / "run_20261125T140433934010Z_4edb90e7_full" / "execution_log.jsonl").open().readline())
    parity_question = e8_first["request"]["messages"][1]["content"][0]["text"].removeprefix("Question: ").removesuffix("\n\nPage images:")
    parity_cache = FINAL_CACHE / "parity"
    for sub in ("pages", "queries", "vision_features"):
        (parity_cache / sub).mkdir(parents=True, exist_ok=True)
    pass1(parity_jobs, parity_cache)
    pass2(parity_jobs, [parity_question], parity_cache)
    checks = {}
    for job in parity_jobs:
        new = load_file(str(parity_cache / "pages" / f"{job['sha256']}.safetensors"))["embedding"]
        old = load_file(str(E5_CACHE_DIR / "pages" / f"{job['sha256']}.safetensors"))["embedding"]
        checks[f"page {job['doc_id'][:30]} p.{job['page']}"] = bool(new.dtype == old.dtype and new.shape == old.shape and torch.equal(new, old))
    qkey = sha256_bytes(parity_question.encode())
    new_q, old_q = (load_file(str(c / "queries" / f"{qkey}.safetensors"))["embedding"] for c in (parity_cache, E5_CACHE_DIR))
    checks["question (E8's first)"] = bool(new_q.shape == old_q.shape and torch.equal(new_q, old_q))
    atomic_write_bytes(PARITY_FILE, json.dumps({"passed": all(checks.values()), "checks": checks, "encoder": ENCODER_CONFIG}, indent=1).encode())
    display(pd.Series(checks, name="bitwise equal to E5").to_frame())
    assert all(checks.values()), "PARITY GATE FAILED: the encoder does not reproduce E5; no pool page may be encoded"
    print("Parity gate passed: fresh encodings equal the frozen E5 cache bit for bit.")
elif PARITY_FILE.is_file():
    print("Parity gate (recorded):", json.loads(PARITY_FILE.read_text())["checks"])

In [8]:
# Encode every pool page (index mode only), 200 pages per chunk: pass 1 then pass 2, so transient features stay below ~0.4 GB.
if MODE == "index":
    assert json.loads(PARITY_FILE.read_text())["passed"], "parity gate not passed"
    assert pool_pages is not None and pool_pages["doc_id"].nunique() == len(POOL_DOCS), "render and text extraction are incomplete"
    jobs = pool_pages.drop_duplicates("sha256")[["doc_id", "page", "sha256", "image_path"]].to_dict("records")
    pending = [j for j in jobs if not (FINAL_CACHE / "pages" / f"{j['sha256']}.safetensors").is_file()]
    print(f"{len(jobs)} distinct page images, {len(pending)} to encode.", flush=True)
    size, started = SETTINGS["encode_chunk_pages"], time.perf_counter()
    for start in range(0, len(pending), size):
        chunk = pending[start:start + size]
        pass1(chunk, FINAL_CACHE)
        pass2(chunk, [], FINAL_CACHE)
        done = start + len(chunk)
        rate = (time.perf_counter() - started) / done
        print(f"  encoded {done}/{len(pending)} pages, {rate:.1f} s/page, about {rate * (len(pending) - done) / 3600:.1f} h left", flush=True)
    assert not any((FINAL_CACHE / "vision_features").glob("*.safetensors")), "unconsumed pass-1 features remain"

if pool_pages is not None and FINAL_CACHE.is_dir():
    encoded = pool_pages["sha256"].map(lambda s: (FINAL_CACHE / "pages" / f"{s}.safetensors").is_file())
    print(f"Pool pages with an embedding: {int(encoded.sum())}/{len(pool_pages)}.")

In [9]:
# The index manifest: every pool page's image hash and embedding hash. It is committed BEFORE any question is loaded.
MANIFEST = FINAL_DIR / "index_manifest.csv"
FROZEN_INDEX = PROJECT_ROOT / "experiments" / "frozen" / "final_index"
if pool_pages is not None and FINAL_CACHE.is_dir():
    complete = pool_pages["doc_id"].nunique() == len(POOL_DOCS) and \
        pool_pages["sha256"].map(lambda s: (FINAL_CACHE / "pages" / f"{s}.safetensors").is_file()).all()
    if complete and not MANIFEST.is_file():
        manifest = pool_pages[["doc_id", "page", "native_words", "ocr_used", "width_px", "height_px", "sha256"]].copy()
        manifest["embedding_sha256"] = manifest["sha256"].map(lambda s: sha256_file(FINAL_CACHE / "pages" / f"{s}.safetensors"))
        manifest["text_sha256"] = [sha256_bytes(t.encode()) for t in pool_pages["text"]]
        atomic_write_bytes(MANIFEST, manifest.to_csv(index=False).encode())
        print(f"Index manifest written: {len(manifest)} pages. Next: freeze it in experiments/frozen/final_index and commit.")
INDEX_FROZEN = (FROZEN_INDEX / "index_manifest.csv").is_file() and not git("status", "--porcelain", "--", str(FROZEN_INDEX.relative_to(PROJECT_ROOT))).strip() \
    and bool(git("ls-files", "--", str((FROZEN_INDEX / "index_manifest.csv").relative_to(PROJECT_ROOT))).strip())
print(f"Index frozen and committed: {INDEX_FROZEN}. Questions {'may now be loaded' if INDEX_FROZEN else 'stay unread'}.")

Index frozen and committed: False. Questions stay unread.
